# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### My feature vector

My decision moment is the end of March 2026. I therefore use only March observations to construct the feature vector.

The unit of analysis is one client-content pair at the March decision point.

My five features are:

1. `gsc_impressions` — March search visibility.
2. `gsc_clicks` — March search clicks.
3. `gsc_avg_position` — March average search position.
4. `sessions_organic` — March organic sessions.
5. `ga4_engaged_sessions` — March engaged sessions.

I aggregate daily observations to the March client-content level. I require `gsc_data_available IS TRUE` so unavailable Search Console data is not treated as real zero performance.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q duckdb scikit-learn
import duckdb
import pandas as pd

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/**/*.parquet"
)

print("Warehouse connection ready.")

Warehouse connection ready.


In [5]:
march_features_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet('{march_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

march_features = con.sql(march_features_query).df()

march_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,1.0,0.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `gsc_impressions` | Search impressions during March | Kept as observed; unavailable Search Console rows are filtered out | Available at the end of March |
| `gsc_clicks` | Search clicks during March | Kept as observed; unavailable Search Console rows are filtered out | Available at the end of March |
| `gsc_avg_position` | Average Search Console position during March | Kept as observed; missing values remain missing | Available at the end of March |
| `sessions_organic` | Organic Analytics sessions during March | Missing/unsupported GA4 observations are not interpreted as genuine zeroes | Available at the end of March |
| `ga4_engaged_sessions` | Engaged Analytics sessions during March | Missing/unsupported GA4 observations are not interpreted as genuine zeroes | Available at the end of March |

The identifiers `client_hash_id` and `content_hash_id` are context fields, not model features. They are used to group and join observations.

No categorical feature is needed for this first leakage check.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]

missingness = (
    march_features[feature_columns]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
)

print("Missing percentage by feature:")
missingness

Missing percentage by feature:


,0
ga4_engaged_sessions,27.57
sessions_organic,27.57
gsc_impressions,0.00
gsc_avg_position,0.00
gsc_clicks,0.00


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### The future outcome

I use April 2026 only to construct the outcome after the March decision point.

My provisional label is whether March-to-April Google Search impressions declined by more than 20%.

This is a warehouse-derived proxy, not a column that already exists in the warehouse. The April information is therefore label information and must never be included among March features.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
april_outcome_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS april_impressions

FROM read_parquet('{april_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcome = con.sql(april_outcome_query).df()

april_outcome.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,april_impressions
0,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,26.0
1,client_62f4a7e64f5e0096,content_f1c085e5ea530266,64.0
2,client_62f4a7e64f5e0096,content_c4901b51a12b1c3b,81.0
3,client_62f4a7e64f5e0096,content_e68b30ce53db783f,89.0
4,client_62f4a7e64f5e0096,content_dd37ba6019bb33f7,34.0


In [8]:
model_frame = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_frame.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions,april_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0,6787.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,405.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,0.0,8475.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,1.0,0.0,6091.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,0.0,0.0,287.0


In [9]:
model_frame["impression_change_pct"] = (
    (
        model_frame["april_impressions"]
        - model_frame["gsc_impressions"]
    )
    / model_frame["gsc_impressions"]
) * 100

In [10]:
model_frame["decline_label"] = (
    model_frame["impression_change_pct"] < -20
).astype(int)

In [11]:
print("Rows with March + April data:", len(model_frame))

print("\nLabel counts:")
print(model_frame["decline_label"].value_counts())

print("\nLabel proportions:")
print(model_frame["decline_label"].value_counts(normalize=True).round(3))

Rows with March + April data: 158549

Label counts:
decline_label
0    82738
1    75811
Name: count, dtype: int64

Label proportions:
decline_label
0    0.522
1    0.478
Name: proportion, dtype: float64


### Deliberate leakage test

I deliberately create `LEAKED_FUTURE_LABEL`, which is exactly the target derived from April data.

This is intentionally wrong.

At the March decision moment, April performance does not exist yet. Therefore this column would not be available to a real decision-maker.

I expect a model given this feature to achieve an unrealistically strong score because the feature directly contains the answer.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_frame["LEAKED_FUTURE_LABEL"] = model_frame["decline_label"]

leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions",
    "LEAKED_FUTURE_LABEL"
]

leaky_data = model_frame.dropna(
    subset=leaky_features + ["decline_label"]
)

X_leaky = leaky_data[leaky_features]
y_leaky = leaky_data["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.2,
    random_state=42,
    stratify=y_leaky
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_predictions
)

print("Accuracy WITH leakage:", round(leaky_accuracy, 4))

Accuracy WITH leakage: 1.0


In [13]:
model_frame = model_frame.drop(
    columns=["LEAKED_FUTURE_LABEL"]
)

In [14]:
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]

honest_data = model_frame.dropna(
    subset=honest_features + ["decline_label"]
)

X_honest = honest_data[honest_features]
y_honest = honest_data["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y_honest,
    test_size=0.2,
    random_state=42,
    stratify=y_honest
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_predictions
)

print("Accuracy WITHOUT leakage:", round(honest_accuracy, 4))

Accuracy WITHOUT leakage: 0.5529


### What the leakage experiment taught me

The model performed much better when `LEAKED_FUTURE_LABEL` was included.

That improvement is not evidence of a better model. It is evidence that the model received information from the future that would not exist at the March decision moment.

After removing the leaked feature, the model produced the honest score.

The key rule I will carry forward is:

**A feature is valid only if it would have been available at the exact moment the prediction or decision was made.**

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

**`client_hash_id`** — Excluded from model features because it is an identifier used for grouping and joining, not a meaningful behavioral feature.

**`content_hash_id`** — Excluded because it identifies the content item and should not be used to let the model memorize individual items.

**April performance fields** — Excluded because April occurs after the March decision moment and therefore represents future information.

**`decline_label`** — Excluded because it is the target being predicted.

**`impression_change_pct`** — Excluded because it is calculated using April impressions and therefore directly contains future outcome information.

**`LEAKED_FUTURE_LABEL`** — Created only for the deliberate leakage experiment and deleted afterward. It must never be part of the final feature set.

**GA4 unavailable observations** — Not treated as genuine zero performance because the warehouse documentation states that GA4 values can be zero-filled before a client's GA4 start date. I use `ga4_data_available` to distinguish availability from actual zero activity.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Final feature columns:")

for col in honest_features:
    print("-", col)

print("\nLeak column still present:",
      "LEAKED_FUTURE_LABEL" in model_frame.columns)

Final feature columns:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- sessions_organic
- ga4_engaged_sessions

Leak column still present: False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.